# Limpieza y Procesamiento del Dataset SQuAD 2.0
Este notebook realiza la limpieza de caracteres UTF-8, normalización de espacios, extracción de respuestas principales, remoción de duplicados y cálculo de métricas de longitud del dataset.

In [1]:
import os
import json
import re
import urllib.request
import pandas as pd

def limpiar_texto(texto: str) -> str:
    if not isinstance(texto, str):
        return ""
    t = re.sub(r"[	 ​]+", " ", texto)
    t = re.sub(r"[ ]+", " ", t)
    t = re.sub(r"
{3,}", "

", t)
    return t.strip()

# Localizar o descargar train-v2.0.json
json_path = "inputs/train-v2.0.json" if os.path.exists("inputs/train-v2.0.json") else "train-v2.0.json"
if not os.path.exists(json_path):
    url = "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json"
    print(f"Descargando {url}...")
    os.makedirs("inputs", exist_ok=True)
    urllib.request.urlretrieve(url, "inputs/train-v2.0.json")
    json_path = "inputs/train-v2.0.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

raw_rows = []
for article in data.get("data", []):
    title = limpiar_texto(article.get("title", ""))
    for paragraph in article.get("paragraphs", []):
        context = limpiar_texto(paragraph.get("context", ""))
        for qa in paragraph.get("qas", []):
            question = limpiar_texto(qa.get("question", ""))
            qa_id = qa.get("id", "").strip()
            is_impossible = qa.get("is_impossible", False)
            answers = qa.get("answers", [])
            primary_answer = ""
            if answers and isinstance(answers, list) and len(answers) > 0:
                primary_answer = limpiar_texto(answers[0].get("text", ""))
            raw_rows.append({
                "title": title,
                "context": context,
                "question": question,
                "id": qa_id,
                "is_impossible": is_impossible,
                "primary_answer": primary_answer,
                "has_answer": not is_impossible and len(primary_answer) > 0
            })

df_raw = pd.DataFrame(raw_rows)
print(f"Total filas crudas leídas: {len(df_raw)}")
df_raw.head()


Descargando / Leyendo SQuAD 2.0...
Total filas crudas leídas: 130319


In [2]:
# Filtrar registros vacíos y eliminar duplicados de (contexto, pregunta)
df_clean = df_raw[(df_raw["context"].str.len() > 20) & (df_raw["question"].str.len() > 5)].copy()
df_clean = df_clean.drop_duplicates(subset=["context", "question"]).copy()
print(f"Filas originales: {len(df_raw)}")
print(f"Filas limpias y deduplicadas: {len(df_clean)}")
df_clean.head()


Filas originales: 130319
Filas limpias y deduplicadas: 130232


In [3]:
df_clean["context_words"] = df_clean["context"].apply(lambda c: len(c.split()))
df_clean["question_words"] = df_clean["question"].apply(lambda q: len(q.split()))

df_por_contexto = (
    df_clean.groupby("context")["question"]
    .apply(list)
    .reset_index(name="questions")
)
df_por_contexto["num_questions"] = df_por_contexto["questions"].apply(len)
df_por_contexto["context_words"] = df_por_contexto["context"].apply(lambda c: len(c.split()))

print(f"Total contextos únicos: {len(df_por_contexto)}")
print(f"Promedio palabras por contexto: {df_clean['context_words'].mean():.1f}")
print(f"Promedio palabras por pregunta: {df_clean['question_words'].mean():.1f}")
df_por_contexto.head()


Total contextos únicos: 19029
Promedio palabras por contexto: 119.6
Promedio palabras por pregunta: 9.9


In [4]:
os.makedirs("outputs/processed", exist_ok=True)
df_clean.to_csv("resultadoBase.csv", index=False, encoding="utf-8")
df_clean.to_csv("outputs/processed/resultadoBase.csv", index=False, encoding="utf-8")
df_por_contexto.to_csv("outputs/processed/resultadoPorContexto.csv", index=False, encoding="utf-8")
print("✅ Exportación completada:")
print(" - resultadoBase.csv")
print(" - outputs/processed/resultadoBase.csv")
print(" - outputs/processed/resultadoPorContexto.csv")


✅ Exportación completada:
 - resultadoBase.csv
 - outputs/processed/resultadoBase.csv
 - outputs/processed/resultadoPorContexto.csv
